### Programming for Biomedical Informatics
#### Week 3 - Data Integration & Summary Analysis

Using some of the skills we've developed working with eUtils we're now going to take two different lists of genes that use different identifiers convert them to NCBI Gene IDs and then use these to merge the data together. With the final merged data we will do some calculations and plots.

In [26]:
# Preliminaries
from Bio import Entrez
import urllib.request
import json
import xml.etree.ElementTree as ET
import pandas as pd

# load my API key from the file
with open('../../bio_api_keys/ncbi.txt', 'r') as file:
    api_key = file.read().strip()

with open('../../bio_api_keys//ncbi_email.txt', 'r') as file:
    email = file.read().strip()

Entrez.api_key = api_key
Entrez.email = email

In [28]:
# Step 1 - Load the two lists that we cannot currenly combine

'''The first file contains a list of gene symbols

e.g.
GeneSymbol
ADAM10
ADAM17
APP
NAE1
APBB1
GAPDH
BACE1

The second file contains a list of RefSeq transcripts (mRNA), and their associated GO terms:

 #NC,NG,NP is refseq - rember NM is NMRNA
e.g.
RefSeqID        GOTerm  Description
NM_001320570    GO:0003824      catalytic activity # Makes protein that catalyses a reaction
NM_001320570    GO:0016787      hydrolase activity
NM_001320570    GO:0140096      catalytic activity, acting on a protein
NM_001320570    GO:0043226      organelle
NM_001320570    GO:0005634      nucleus
NM_001320570    GO:0005794      Golgi apparatus

We are going to convert Gene Symbols and Refseq IDs to NCBI Gene IDs, and then combine the two lists into a single table.
'''

# Load the gene symbols as a pandas dataframe
'''### YOUR CODE HERE ###'''
gene_df = pd.read_csv('./GeneSymbols.tsv', sep = '\t')


# Load the RefSeq data as a pandas dataframe
'''### YOUR CODE HERE ###'''
refseq_df = pd.read_csv('./transcript_functions.tsv', sep = '\t')

In [29]:
# view the first few rows of each dataframe
'''### YOUR CODE HERE ###'''
gene_df.head()

,GeneSymbol
0,ADAM10
1,ADAM17
2,APP
3,NAE1
4,APBB1


In [30]:
# view the first few rows of each dataframe
'''### YOUR CODE HERE ###'''
refseq_df.head()

,RefSeqID,GOTerm,Description
0,NM_001320570,GO:0003824,catalytic activity
1,NM_001320570,GO:0016787,hydrolase activity
2,NM_001320570,GO:0140096,"catalytic activity, acting on a protein"
3,NM_001320570,GO:0043226,organelle
4,NM_001320570,GO:0005634,nucleus


In [31]:
# Making auxiliary get gene id from gene symbol:

def get_gene_id(gene_iter, organism = 'Homo sapians'):
    gene_id_res = {}
    for gene in gene_iter:
        handle = Entrez.esearch(db = 'gene', term= f'{gene}[gene] AND {organism}[Organism]', retmax = 1)
        record = Entrez.read(handle)
        if record['IdList']:
            gene_id_res[gene] = record['IdList'][0]
        else:
            gene_id_res[gene] = None

    return gene_id_res



## Replicate his approach



In [34]:
from math import e
import urllib.parse
from numpy import record
# the base request url for eSearch
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

db = 'gene'

gene_iter = gene_df['GeneSymbol'].tolist()

db_call_string = ' OR '.join([f'{gene}[Gene]' for gene in gene_iter])
db_call_string = f'(human[Organism]) AND ({db_call_string})' #Joining in the query is () AND/OR ()

db_call_string

'(human[Organism]) AND (ADAM10[Gene] OR ADAM17[Gene] OR APP[Gene] OR NAE1[Gene] OR APBB1[Gene] OR GAPDH[Gene] OR BACE1[Gene] OR BACE2[Gene] OR RTN3[Gene] OR RTN4[Gene] OR PSENEN[Gene] OR PSEN1[Gene] OR PSEN2[Gene] OR NCSTN[Gene] OR APH1A[Gene] OR APH1B[Gene] OR IDE[Gene] OR MME[Gene] OR MAPT[Gene] OR ND1[Gene] OR ND2[Gene] OR ND3[Gene] OR ND4[Gene] OR ND4L[Gene] OR ND5[Gene] OR ND6[Gene] OR NDUFV1[Gene] OR NDUFV2[Gene] OR NDUFV3[Gene] OR NDUFA1[Gene] OR NDUFA2[Gene] OR NDUFA3[Gene] OR NDUFA4[Gene] OR NDUFA4L2[Gene] OR NDUFA5[Gene] OR NDUFA6[Gene] OR NDUFA7[Gene] OR NDUFA8[Gene] OR NDUFA9[Gene] OR NDUFA10[Gene] OR NDUFAB1[Gene] OR NDUFA11[Gene] OR NDUFA12[Gene] OR NDUFA13[Gene] OR NDUFB1[Gene] OR NDUFB2[Gene] OR NDUFB3[Gene] OR NDUFB4[Gene] OR NDUFB5[Gene] OR NDUFB6[Gene] OR NDUFB7[Gene] OR NDUFB8[Gene] OR NDUFB9[Gene] OR NDUFB10[Gene] OR NDUFB11[Gene] OR NDUFS1[Gene] OR NDUFS2[Gene] OR NDUFS3[Gene] OR NDUFS4[Gene] OR NDUFS5[Gene] OR NDUFS6[Gene] OR NDUFS7[Gene] OR NDUFS8[Gene] OR NDUFC

In [43]:


# Define the parameters for the eSearch request
# This can be nicely done using a dictionary
# Note we include the history feature of eUtils to allow us to make large queries efficiently
import urllib.request


esearch_params = {
    'db': db,
    'term':db_call_string,
    'api_key': api_key,
    'email': email,
    'usehistory': 'y'
}
encoded_data = urllib.parse.urlencode(esearch_params).encode('utf-8')

# the base request url for eSearch
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

# make the request
request = urllib.request.Request(url, data=encoded_data)
response = urllib.request.urlopen(request)

# read into an XML object
esearch_data_XML = ET.fromstring(response.read())

for child in esearch_data_XML:
    print(child)

# Extract WebEnv and QueryKey
# Here we use ElementTree to extract the WebEnv and QueryKey from the XML response
# We will use these to fetch the gene ids in the next step using eSummary
webenv = esearch_data_XML.find('WebEnv').text
query_key = esearch_data_XML.find('QueryKey').text
count = esearch_data_XML.find('Count').text

print('webenv:', webenv, 'query_key:', query_key, 'count:', count)


<Element 'Count' at 0x000002CCF0713100>
<Element 'RetMax' at 0x000002CCF0713330>
<Element 'RetStart' at 0x000002CCF0712E30>
<Element 'QueryKey' at 0x000002CCF0712840>
<Element 'WebEnv' at 0x000002CCF07127A0>
<Element 'IdList' at 0x000002CCF0712D90>
<Element 'TranslationSet' at 0x000002CCF072CEA0>
<Element 'TranslationStack' at 0x000002CCF072CF90>
<Element 'QueryTranslation' at 0x000002CCF082B7E0>
webenv: MCID_68fb96e10073de9fff06eb48 query_key: 1 count: 449


NameError: name 'query_key' is not defined

In [77]:
# use eSearch to convert gene symbols to NCBI Gene IDs (for the first 10 gene symbols)
# remembering to add API key and email
# remembering to use the [Gene] field in the search
# remembering to specify human
# show the progress by printing the gene symbol and gene ID and the number of gene symbols processed so far
# This takes about 3 minutes (NB not the quickest way!)

esearch_params_2 = {
    'db': db,
    'query_key': query_key,
    'WebEnv': webenv,
    'api_key': api_key,
    'email': email
}

# the base request url for eSummary
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

encoded_data_2 = urllib.parse.urlencode(esearch_params_2).encode('utf-8')

# make the request
request = urllib.request.Request(url, data=encoded_data_2)
response = urllib.request.urlopen(request)

# read into an XML object
esummary_data_XML = ET.fromstring(response.read())

gene_symbols_to_id = {}
for child in esearch_data_XML.findall('DocumentSummarySet/DocumentSummary'):
    print(child.find('Name').text)
    print(child.attrib['uid'])
    gene_symbols_to_id[child.find('Name').text] = child.attrib['uid']

# convert the gene_ids dictionary to a pandas dataframe
gene_ids_df = pd.DataFrame(gene_symbols_to_id.items(), columns=['GeneSymbol', 'GeneID'])

# look at the first few rows
gene_ids_df.head()

# take a look
# import xml.dom.minidom

# xml_str = ET.tostring(esummary_data_XML, encoding='utf-8')
# dom = xml.dom.minidom.parseString(xml_str)
# print(dom.toprettyxml())




APOE
348
TNF
7124
IL6
3569
APP
351
KRAS
3845
BRAF
673
AKT1
207
NFKB1
4790
IL1B
3553
CTNNB1
1499
SNCA
6622
PTGS2
5743
MAPT
4137
MTOR
2475
MAPK1
5594
PIK3CA
5290
RELA
5970
GSK3B
2932
FAS
355
AGER
177
INS
3630
CASP3
836
APC
324
MAPK3
5595
PSEN1
5663
NOS2
4843
IL1A
3552
LPL
4023
CASP8
841
HRAS
3265
MAPK8
5599
INSR
3643
TNFRSF1A
7132
CSNK2A1
1457
BECN1
8678
PIK3R1
5295
NRAS
4893
DKK1
22943
IRS1
3667
RAF1
5894
WNT5A
7474
ADAM17
6868
LRP1
4035
BACE1
23621
PROC
5624
CACNA1C
775
FASN
2194
CALM1
801
IKBKB
3551
EIF2AK2
5610
MME
4311
CYBB
1536
NOX4
50507
NOS1
4842
IKBKG
8517
CHRNA7
1139
GAPDH
2597
MAP2K1
5604
ADAM10
102
CASP9
842
CHUK
1147
GRIN2B
2904
CDK5
1020
TRAF2
7186
XBP1
7494
LRP5
4041
GNAQ
2776
ATF4
468
ERN1
2081
ATP2A2
488
AKT2
208
CSF1
1435
SDHB
6390
VDAC1
7416
FADD
8772
PTGS1
5742
DDIT3
1649
TUBB3
10381
GRIN1
2902
MAP3K5
4217
WNT1
7471
EIF2S1
1965
ULK1
8408
BAD
572
IRS2
8660
GRIN2A
2903
LRP6
4040
PSEN2
5664
EIF2AK3
9451
AXIN1
8312
CSNK2B
1460
NOX1
27035
ITPR1
3708
WNT3A
89780
CAPN1
823
R

,GeneSymbol,GeneID
0,APOE,348
1,TNF,7124
2,IL6,3569
3,APP,351
4,KRAS,3845


In [82]:
gene_ids_df.sort_values(by = 'GeneSymbol', inplace=True)
gene_ids_df.dropna(inplace=True)


In [84]:
gene_ids_df.head()

,GeneSymbol,GeneID
58,ADAM10,102
41,ADAM17,6868
177,ADRM1,11047
19,AGER,177
6,AKT1,207


In [ ]:
# Step 3 - Convert RefSeq IDs to NCBI Gene IDs

# create a dictionary that maps RefSeq IDs to Gene IDs
'''### YOUR CODE HERE ###'''

# use eSearch to convert RefSeq IDs to NCBI Gene IDs (for the first 10 RefSeq IDs)
# remembering to add API key and email
# remembering to use the [Gene] field in the search
# remembering to specify human

# Search for the gene information using the RefSeq transcript ID
'''### YOUR CODE HERE ###'''

# for the first unique 10 values in the refseq dataframe, get the gene ID
# show the progress by printing the refseq ID and gene ID and the number of refseq IDs processed so far
'''### YOUR CODE HERE ###'''

# Note, may need to encode('utf-8')

In [ ]:
#user PrettyTable to display the results
'''### YOUR CODE HERE ###'''

In [ ]:
# Step 4  - merge the refseq_to_gene_id dictionary with the refseq dataframe

# create a new column in the refseq dataframe called 'GeneID'
# fill the column with the gene IDs from the refseq_to_gene_id dictionary
'''### YOUR CODE HERE ###'''

# remove rows with missing values
'''### YOUR CODE HERE ###'''

# display the refseq dataframe
'''### YOUR CODE HERE ###'''

In [ ]:
#Step 7 - combine the gene_symbol and refseq dataframes

# convert the gene_symbol_to_id dictionary to a dataframe
'''### YOUR CODE HERE ###'''

# merge the refseq and gene_symbol_to_id_df dataframes on the 'GeneID' column
'''### YOUR CODE HERE ###'''

# drop the GeneID column
'''### YOUR CODE HERE ###'''

# display the combined dataframe
'''### YOUR CODE HERE ###'''

In [ ]:
#Step 8 - Do some basic summary analysis

# display the number of rows and columns in the combined dataframe
'''### YOUR CODE HERE ###'''

# count how many unique genes are in the combined dataframe
'''### YOUR CODE HERE ###'''

# count how many unique GO terms are in the combined dataframe
'''### YOUR CODE HERE ###'''

# display the number of unique genes and GO terms
'''### YOUR CODE HERE ###'''

# use some plots to visualise the data (up to you!)
'''### YOUR CODE HERE ###'''